In [ ]:
from thbsplines.hierarchical_space import HierarchicalSpace
import numpy as np
import dolfinx
import basix.ufl
import pyvista
from mpi4py import MPI

from dolfinx.jit import ffcx_jit
from dolfinx import default_real_type, default_scalar_type
rtype = default_real_type
dtype = default_scalar_type
import ufl
from ffcx.codegeneration.utils import numba_ufcx_kernel_signature as ufcx_signature

from thbsplines.refinement import refine
from thbsplines.fenicsx.mesh import build_mesh
from thbsplines.fenicsx.functionspace import build_dofmap, fill_function_space, create_spline_space
from thbsplines.fenicsx.solvers import solve_problem
from thbsplines.fenicsx.adaptivity import dorfler_marking
from thbsplines.fenicsx.kernels import make_linear_kernel, make_bilinear_kernel

In [ ]:
p0 = 2
m=3
n_refinements = 0
knots1 = np.array([-1,0.,1], dtype=np.float64)
knots1 = refine(knots1, p=p0, n_times=n_refinements)
log_initial_mesh_size = np.log2(np.max(np.diff(knots1)))
knots2 =refine(np.array([-1,0, 1.], dtype=np.float64), p0, n_times=n_refinements)
err_cells = {}
hs = HierarchicalSpace(knots=[knots1, knots2], degrees=[p0])

In [ ]:
# mesh_data = hs.hmesh.aelem_level
# h0 = 1.            # Base side-length at Level 0
# domain_min = -1.0   # Lower bound of your domain
# domain_width = 2.0  # Total width of your domain (from -1 to 1 is 2.0)
# N0 = int(domain_width / h0) # Number of base cells per side (4 in this case)

# with open("mesh_data_p2_smooth_tanh_step9.dat", "w") as f_write:
#     # Header for PGFPlots
#     f_write.write("x y\n")
    
#     for level, indices in mesh_data.items():
#         # Number of cells along an axis at the current level
#         NL = N0 * (2 ** level)
#         # Element side-length at the current level
#         hL = h0 / (2 ** level)
        
#         for idx in indices:
#             # Lexicographical decoding to grid coordinates
#             x_idx = idx // NL
#             y_idx = idx % NL
            
#             # Calculate physical coordinate boundaries shifted by domain_min
#             xmin = domain_min + (x_idx * hL)
#             ymin = domain_min + (y_idx * hL)
#             xmax = xmin + hL
#             ymax = ymin + hL
            
#             # Write out the 5 vertices to close the square path
#             f_write.write(f"{xmin:.6f} {ymin:.6f}\n")
#             f_write.write(f"{xmax:.6f} {ymin:.6f}\n")
#             f_write.write(f"{xmax:.6f} {ymax:.6f}\n")
#             f_write.write(f"{xmin:.6f} {ymax:.6f}\n")
#             f_write.write(f"{xmin:.6f} {ymin:.6f}\n")
#             f_write.write("\n") # CRITICAL: Tells PGFPlots to lift the pen and start a new box

# print("mesh_data.dat generated successfully!")

In [ ]:
for level, cells in err_cells.items():
    hs.refine(cells, level, refine_neighbours=False, refine_T_neighbours=True, m=m)
hs.hmesh.plot_cells()

In [ ]:
disconnected_mesh, thb_operators, N_max, _ = build_mesh(hs=hs)

In [ ]:
legendre_elt = basix.ufl.element(
    "DG",
    "quadrilateral",
    degree=p0,
    lagrange_variant=basix.LagrangeVariant.legendre
)
V = dolfinx.fem.functionspace(disconnected_mesh, legendre_elt)
print(f"Number of degrees of freedom: {V.dofmap.index_map.size_global}")
dx_custom = ufl.Measure("dx", domain=disconnected_mesh, metadata={"quadrature_degree": 12})
u,v = ufl.TrialFunction(V), ufl.TestFunction(V) 
my_x = ufl.SpatialCoordinate(disconnected_mesh)
# f = dolfinx.fem.Function(V)
# f.interpolate(lambda x: (np.tanh(9*x[1]-9*x[0])+1)/9. + 1./(1.5*np.exp((10.*x[0]-6.)**2 + 
#                                                                        (10.*x[1]+7)**2)) + 
#                                                                        1./(np.exp(np.sqrt((2*x[0]+1)**2 + (2*x[1]-1)**2))))
#f = my_x[0]*my_x[1]*(knots1[-1]-my_x[0])*(knots2[-1]-my_x[1])**2
sigma = 0.4
#f = (1./(3.141592*sigma**4))*(1.-0.5*((my_x[0]**2+my_x[1]**2)/sigma**2))*ufl.exp(-(my_x[0]**2+my_x[1]**2)/(2.*sigma**2))
#f = (ufl.tanh(9.*(my_x[1]-my_x[0]))+1)/9. + 2./3*ufl.exp(-ufl.sqrt((10.*my_x[0]-6.)**2+(10.*my_x[1]+7.)**2))
f = (ufl.tanh(9.*(my_x[1]-my_x[0]))+1)/9. + (2./3.)*ufl.exp(-(10.*my_x[0]-6.)**2-(10.*my_x[1]+7.)**2)
a0 = ufl.inner(u, v) * dx_custom
f0 = ufl.inner(f, v)*dx_custom

f_square_integral = dolfinx.fem.assemble_scalar(dolfinx.fem.form(ufl.inner(f,f)*dx_custom))
f_sq_integral = np.sqrt(disconnected_mesh.comm.allreduce(f_square_integral, op=MPI.SUM))

msh = disconnected_mesh
ufcxa0, _, _ = ffcx_jit(msh.comm, a0, form_compiler_options={"scalar_type": dtype})  # type: ignore
kernela0 = getattr(ufcxa0.form_integrals[0], f"tabulate_tensor_{np.dtype(dtype).name}")  # type: ignore

ufcxf0, _, _ = ffcx_jit(msh.comm, f0, form_compiler_options={"scalar_type": dtype})  # type: ignore
kernelf0 = getattr(ufcxf0.form_integrals[0], f"tabulate_tensor_{np.dtype(dtype).name}")  # type: ignore

In [ ]:
dofmap, padded_cells_to_dofs = build_dofmap(hierarchical_space=hs, mesh=disconnected_mesh, 
                                            N_max=N_max, morton=True)
C_func, C_space = fill_function_space(hierachical_space=hs, mesh=disconnected_mesh,
                                      N_max=N_max, thb_operators=thb_operators)
V_spline = create_spline_space(cells_to_dofs=padded_cells_to_dofs, mesh=disconnected_mesh,
                               N_max=N_max, mult_factor=1)

In [ ]:
local_dofs = (hs.degrees[0]+1)**2
tabulate_A = make_bilinear_kernel(dtype, rtype, ufcx_kernel=kernela0, padded_dofs=N_max, local_dofs=local_dofs)
tabulate_b = make_linear_kernel(dtype, rtype, ufcx_kernel=kernelf0, padded_dofs=N_max, local_dofs=local_dofs)

In [ ]:
formtype = dolfinx.fem.form_cpp_class(dtype)  # type: ignore
# Gets the number of cells for which each individual core is responsible for.
cells = np.arange(msh.topology.index_map(msh.topology.dim).size_local, dtype=np.int32)

# The 4th argument np.array([...], dtype=np.int8) is the 
# active coefficients array. It lists which indices from the 
# coefficients list should be packed into the w_ pointer that the kernel receives.
integrals = {dolfinx.fem.IntegralType.cell: [
    (0, tabulate_A.address, cells, np.array([0], dtype=np.int8))]}

a_cond = dolfinx.fem.Form( # We are not forming anything yet, this is a recipe
    formtype( # selectes the correct floating-point precision
        spaces=[V_spline._cpp_object, 
                V_spline._cpp_object]
            , # trial and test spaces, determines the size of A_
        integrals=integrals, #this is a dictionary, and we are passing the adress of tabulate_A() here
        coefficients=[C_func._cpp_object
                    ], # weights w_, holds C@T
              constants=[],
              need_permutation_data=False,
              entity_maps=[], 
              mesh=msh._cpp_object)
)

integrals_rhs = {dolfinx.fem.IntegralType.cell: [(0, tabulate_b.address, cells, np.array([0], dtype=np.int8))]}
l_cond = dolfinx.fem.Form(
    formtype(
        spaces=[V_spline._cpp_object], # test space, determines the size of b_
        integrals=integrals_rhs, #give the adress of tabulate_b
        coefficients=[C_func._cpp_object], # holds the evaluations of f at the correct points, as well as C@T
        constants=[], need_permutation_data=False, entity_maps=[], mesh=msh._cpp_object
    )
)

In [ ]:
x_vec, A = solve_problem(hs=hs, a=a_cond, rhs=l_cond, dirichlet_indices=None, 
                      dummy_index=np.max(padded_cells_to_dofs), V_spline=V_spline,
                      iterative=False, return_A=True)


In [ ]:
# # Extract the CSR (Compressed Sparse Row) arrays from PETSc
# indptr, indices, data = A.getValuesCSR()

# # Get the global size of the matrix
# shape = A.getSize()

# # # Create a SciPy CSR matrix
# A_scipy = sp.csr_array((data, indices, indptr), shape=shape)
# rows, cols = A_scipy.nonzero()
# with open("nnz_tanh_exp_sqrt_morton.dat", "w") as f_write:
#     for r, c in zip(rows, cols):
#         f_write.write(f"{c+1} {r+1}\n")
# # # print(f"Matrix shape: {A_scipy.shape}")
# # # print(f"Number of non-zeros: {A_scipy.nnz}")

# import matplotlib.pyplot as plt

# plt.figure(figsize=(7, 7))
# # plt.spy plots the non-zero entries of a matrix
# plt.spy(A_scipy, markersize=2, color='darkgray')
# plt.title("Sparsity Pattern of THB-Spline stiffness matrix \n Morton ordering")
# plt.show()

In [ ]:
u_dg = dolfinx.fem.Function(V)
c_values = C_func.x.array.reshape((-1, N_max, (hs.degrees[0]+1)**2))
num_cells_local = disconnected_mesh.topology.index_map(disconnected_mesh.topology.dim).size_local

# Map the global B-spline coefficients back to local Legendre coefficients
for local_idx in range(num_cells_local):
    # Get global B-spline dof indices for this cell
    spline_dofs = padded_cells_to_dofs[local_idx]
    
    # Extract the B-spline coefficients for this cell
    u_spline_local = x_vec[spline_dofs]
    
    # Get the local transformation matrix G for this cell
    G = c_values[local_idx, :, :]
    
    # Transform B-spline to DG: mathematically, the kernel does A = G @ A0 @ G.T
    # This implies the coefficient mapping is u_dg = G.T @ u_spline
    u_dg_local = G.T @ u_spline_local
    
    # Assign to the standard DG function
    dg_dofs = V.dofmap.cell_dofs(local_idx)
    u_dg.x.array[dg_dofs] = u_dg_local

u_dg.x.scatter_forward()


# Compute exact L2 error using FEniCSx standard UFL
error_form = dolfinx.fem.form(ufl.inner(f - u_dg, f - u_dg) * dx_custom)
error_sq = dolfinx.fem.assemble_scalar(error_form)
exact_l2_error = np.sqrt(disconnected_mesh.comm.allreduce(error_sq, op=MPI.SUM))

print(f"Exact L2 Error (via DG projection): {exact_l2_error:.2e}")
rel_err = exact_l2_error/f_sq_integral
print(f"Relative error = {rel_err:.2e}")

In [ ]:
print(f"({x_vec.shape[0]}, {rel_err:.5e})")

In [ ]:
V_error = dolfinx.fem.functionspace(disconnected_mesh, ("DG", 0))
v = ufl.TestFunction(V_error)
local_error_form = dolfinx.fem.form(ufl.inner(f - u_dg, f - u_dg) * v * dx_custom)

err_cells = dorfler_marking(hierarchical_space=hs, theta=0.7, local_error_form=local_error_form)

In [ ]:
# import dolfinx.plot
# import pyvista

# #1. Create a "Nodal" DG space of the same degree for plotting
# #By default, DG with no variant specified uses Lagrange (nodal)
# v_plot_elt = basix.ufl.element(
#     "DG", 
#     "quadrilateral", 
#     degree=p0+2
# )
# V_plot = dolfinx.fem.functionspace(disconnected_mesh, v_plot_elt)

# # 2. Interpolate your computed solution (u_dg) into the nodal space
# u_plot = dolfinx.fem.Function(V_plot)
# #error_ufl = ufl.ln(ufl.sqrt((u_dg-f)**2)+1e-5)
# error_ufl = u_dg
# error_expr = dolfinx.fem.Expression(error_ufl, V_plot.element.interpolation_points)
# u_error = dolfinx.fem.Function(V_plot)
# u_error.interpolate(error_expr)

# # 3. Now use V_plot for the VTK mesh generation
# topology, cell_types, geometry = dolfinx.plot.vtk_mesh(V_plot)
# grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

# # 4. Attach the interpolated values
# grid.point_data["u"] = u_error.x.array.real
# grid.set_active_scalars("u")

# # 5. Plotting (with a 'shrink' to see your disconnected mesh boundaries!)
# plotter = pyvista.Plotter()
# grid_shrink = grid.shrink(0.95) # This makes the "disconnected" nature visible
# plotter.add_mesh(grid_shrink, show_edges=False, cmap="turbo")
# plotter.view_xy()
# plotter.show(jupyter_backend="static")
# plotter.show()

# from pyvista.trame.jupyter import launch_server
# pyvista.set_jupyter_backend('client')
# warped_grid = grid.warp_by_scalar("u", factor=1.) 

# # If you still want to see the gaps between cells:
# grid_shrink = warped_grid.shrink(0.95)

# plotter.add_mesh(grid_shrink, show_edges=False, cmap="viridis", lighting=True)

# # Set a nice 3D camera angle instead of view_xy()
# plotter.camera_position = 'iso' 
# await launch_server().ready
# plotter.show()